# R3-M1 batch 01: full-20 stopping experiments
Use a T4 runtime and the same environment as the original run. Sync the entire updated experiment folder first. Confirm with the team that Protocol A has not begun and B has not been locked; local checks cannot discover unrecorded work. This notebook trains only on B train and selects only on B selection.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
ROOT = Path('/content/drive/MyDrive/PAI&AIES Project')
EXP = ROOT / 'experiments/r3_m1'
SHARDS = ROOT / 'data/processed/cohort_v1'
AUDIT = EXP / 'reports/audit_20261107T165239Z.json'
CONTRACT = EXP / 'configs/batch01_resolved.json'
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src')
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
TEMP = EXP / 'runs/notebook_tmp'
TEMP.mkdir(parents=True, exist_ok=True)
for key in ('TMPDIR','TEMP','TMP'):
    os.environ[key] = str(TEMP)
import torch
assert torch.cuda.is_available(), 'Select a T4 runtime'
assert AUDIT.is_file() and SHARDS.is_dir()
print(torch.__version__, torch.cuda.get_device_name(0))
def run_script(name, *args):
    subprocess.run([sys.executable, str(EXP / 'scripts' / name), *map(str, args)], check=True)


## Tests and contract
A failed or skipped test stops the notebook. On reconnection, the existing immutable contract is reused; do not delete it or change source files to get past a resume rejection.


In [ ]:
check = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', str(EXP / 'tests'), str(ROOT / 'tests/test_r3_bigru_frontend.py')], capture_output=True, text=True)
print(check.stdout, check.stderr)
assert check.returncode == 0 and ' skipped' not in check.stdout
if not CONTRACT.exists():
    run_script('prepare_batch.py', '--root', ROOT, '--audit', AUDIT, '--output', CONTRACT)
else:
    print('Reusing contract:', CONTRACT)


## Two fits in separate processes
The first run uses LR 0.001, the second 0.0003. Both run up to 20 epochs without patience-based stopping. Existing completed runs are checked and reused. A stale active-attempt marker or inconsistent checkpoint files require investigation; do not remove files manually. Do not run another training notebook concurrently against this experiment folder.


In [ ]:
for experiment in ('b01_baseline_full20', 'b01_lower_lr_full20'):
    directory = EXP / 'runs/batch01' / experiment
    args = ['--contract', CONTRACT, '--experiment', experiment, '--shard-dir', SHARDS]
    if directory.exists():
        args += ['--resume']
    print('Running:', experiment, flush=True)
    run_script('train.py', *args)


## Compare and freeze the candidate choice
The comparison requires two complete 20-epoch runs. It recomputes selection metrics from the saved probabilities, reports the hypothetical patience-3 result, and compares against the original models. No test or calibration/operating performance influences the decision.


In [ ]:
run_script('compare.py', '--contract', CONTRACT)
print((EXP / 'reports/batch01_comparison.md').read_text())


## Export for R3-N1
Only now is the chosen model scored on calibration and operating roles. The saved scores remain uncalibrated. Independent review and the later B lock remain pending.


In [ ]:
run_script('export_selected.py', '--contract', CONTRACT, '--shard-dir', SHARDS)
print('Share:', EXP / 'reports/batch01_comparison.json', EXP / 'reports/final_handoff.json', EXP / 'ledger.csv')
print('Keep checkpoints and predictions:', EXP / 'runs/batch01')
